In [1]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import  r2_score, mean_squared_error

import joblib

# Loading the data

In [2]:
path = "../Data/calorie_burned_data.csv"
df = pd.read_csv(path)
print(df.head())
print(df.info())

   id     Sex  Age  Height  Weight  Duration  Heart_Rate  Body_Temp  Calories
0   0    male   36   189.0    82.0      26.0       101.0       41.0     150.0
1   1  female   64   163.0    60.0       8.0        85.0       39.7      34.0
2   2  female   51   161.0    64.0       7.0        84.0       39.8      29.0
3   3    male   20   192.0    90.0      25.0       105.0       40.7     140.0
4   4  female   38   166.0    61.0      25.0       102.0       40.6     146.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          750000 non-null  int64  
 1   Sex         750000 non-null  object 
 2   Age         750000 non-null  int64  
 3   Height      750000 non-null  float64
 4   Weight      750000 non-null  float64
 5   Duration    750000 non-null  float64
 6   Heart_Rate  750000 non-null  float64
 7   Body_Temp   750000 non-null  float64
 8

In [3]:
target_column = "Calories"
X = df.drop(columns=[target_column])
y = df[target_column]

# Feature Engineering

In [ ]:
numeric_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [6]:
df["Gender"] = df["Sex"].map({'male': 0, 'female': 1})

In [15]:
df.head()
df.drop("Sex",axis=1, inplace=True)

In [9]:
df.isnull().sum()
df["Gender"].value_counts()

Gender
1    375721
0    374279
Name: count, dtype: int64

In [16]:
df.head()

,id,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories,Gender
0,0,36,189.0,82.0,26.0,101.0,41.0,150.0,0
1,1,64,163.0,60.0,8.0,85.0,39.7,34.0,1
2,2,51,161.0,64.0,7.0,84.0,39.8,29.0,1
3,3,20,192.0,90.0,25.0,105.0,40.7,140.0,0
4,4,38,166.0,61.0,25.0,102.0,40.6,146.0,1


# Feature Engineering 

Handling Outliers

# Splitting the data 

In [17]:
X = df.drop("Calories",axis=1)
y = df["Calories"]

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
print(X_train.shape)
print(X_test.shape)

(600000, 8)
(150000, 8)


In [20]:
# Scale numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [28]:
model = {
    "Linear Regression": LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=42),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=200,
                                            random_state=42, n_jobs=-1)
}

In [29]:
cv_result = {}
for name, mod in model.items():
    scores = cross_val_score(
        mod, 
        X_train, 
        y_train,
        scoring="neg_root_mean_squared_error", 
        cv=5
    )
    rmse = -scores.mean()
    cv_result[name] = rmse

    print(f"{name}: CV RMSE = {rmse:.4f}")

Linear Regression: CV RMSE = 11.1073
DecisionTreeRegressor: CV RMSE = 5.2602


KeyboardInterrupt: 

In [25]:
pred = model.predict(X_test)

In [26]:
print(r2_score(y_test, pred))
print(mean_squared_error(y_test, pred,squared=False))

9.083284675781567e-05
205.15560948947996


d:\Users\hp\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


# Save & Load the Model and Scaled Features

In [28]:
joblib.dump(model, "../Model/calorie_model.pkl")
joblib.dump(scaler, "../Model/scaler.pkl")

['../Model/scaler.pkl']